# Phases 4-5 - smoke test + pilot (Colab)

Needs the Phase 3 embeddings in Drive (`embed_colab.ipynb`).

1. **Smoke** - one cell (C1, NLI, linear, seed 0); test accuracy should land
   in ~0.75-0.85 for frozen mpnet + a linear head.
2. **Pilot** - 3 seeds x linear head x 10 conditions x 3 datasets = 90 cells.
   Sanity checks only, no statistics (PROTOCOL.md section 17 phase 5).

CPU is fine (a GPU runtime is a little faster). `run_grid.py` is resumable.
`pilot_runs.parquet` is tiny - download it, commit to the repo, analyze locally.

In [ ]:
!git clone https://github.com/ryanteachman/sbert-head-ablation.git
%cd sbert-head-ablation
!pip install -q pyarrow pyyaml scikit-learn
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/sbert-head-ablation'
import os; os.makedirs(f'{DRIVE}/results', exist_ok=True)
# pull embeddings to local disk (fast reads during the grid)
!mkdir -p embeddings results && rsync -a "{DRIVE}/embeddings/" embeddings/
import json; m = json.load(open('embeddings/meta.json'))
print(len(m['splits']), '/ 11 splits'); assert len(m['splits']) == 11, 'run embed_colab.ipynb first'

## 1. Smoke test - one real cell

In [ ]:
!python src/run_grid.py --pilot --embed-dir embeddings \
  --out results/_smoke.parquet --datasets nli --conditions C1 --seeds 0

In [ ]:
import pandas as pd
s = pd.read_parquet('results/_smoke.parquet').iloc[0]
print(f"NLI C1 linear seed0  ->  test_acc={s.test_acc:.4f}  macro_f1={s.test_macro_f1:.4f}"
      f"  [{s.epochs_trained} epochs, {s.wall_s}s]")
assert 0.65 < s.test_acc < 0.90, 'accuracy outside expected frozen-mpnet range - investigate'
print('smoke OK')

## 2. Pilot - 90 cells
Checks: absolute numbers believable; ordering plausible (C1/C2/C3 >~ C0;
C6/C7 < their with-`u,v` counterparts; C4~C1, C9~C2); per-seed spread small;
early stopping fires before the epoch ceiling.

In [ ]:
!python src/run_grid.py --pilot --embed-dir embeddings --out results/pilot_runs.parquet
!rsync -a results/ "{DRIVE}/results/"

In [ ]:
import pandas as pd
df = pd.read_parquet('results/pilot_runs.parquet')
print(df.groupby(['dataset','condition'])['test_acc'].agg(['mean','std']).round(4).to_string())
ceiling = int(((df.epochs_trained == df.epochs_trained.max()) & (~df.early_stopped)).sum())
print(f'\ncells that hit the epoch ceiling without early-stopping: {ceiling}  (want ~0)')
print(f'rows: {len(df)}  (want 90)')
from google.colab import files; files.download('results/pilot_runs.parquet')

## Done
`pilot_runs.parquet` downloaded (also in Drive under `results/`). Drop it in
`results/` locally, commit, and review before the full 600-cell grid.